# 🎯 Customer Segmentation & Behavioral Analytics
### End-to-End Unsupervised Machine Learning, RFM Modeling & Strategic Marketing Playbook
**Author:** Data Analytics Specialist  
**Domain:** Retail & E-Commerce Customer Analytics  
**Algorithms:** K-Means Clustering, Agglomerative Hierarchical Clustering, PCA, RFM Segmentation  

---

## 📌 Executive Summary & Project Objectives
Customer segmentation is the foundational cornerstone of customer-centric marketing and lifecycle management. Rather than treating all customers uniformly, data-driven segmentation empowers organizations to:
1. **Uncover Latent Behavioral Archetypes:** Identify distinct groups of shoppers based on their demographic profiles and transactional behaviors.
2. **Optimize Resource Allocation & ROI:** Concentrate high-touch retention campaigns on high-value champions while designing automated win-back funnels for at-risk segments.
3. **Personalize Product & Channel Messaging:** Recommend the right products via their preferred shopping channel (Mobile App, Web, In-Store) at the ideal discount sensitivity.

### Key Milestones Covered in This Notebook:
- **Phase 1: Multi-Source Data Ingestion & Auditing:** Ingestion from CSV and SQLite database with data integrity checks.
- **Phase 2: Exploratory Data Analysis (EDA):** Univariate, bivariate, and multivariate analysis of demographic and transaction features.
- **Phase 3: Preprocessing & Scaling:** Feature engineering, outlier treatment, and z-score standardization via `StandardScaler`.
- **Phase 4: Optimal Cluster Determination:** Mathematical validation using the **Elbow Method (WCSS)** and **Silhouette Coefficient Analysis** ($k=2$ to $k=10$).
- **Phase 5: K-Means Clustering & Centroid Profiling:** Model fitting, cluster assignment, and centroid interpretation.
- **Phase 6: Dimensionality Reduction via PCA:** Principal Component Analysis for 2D and 3D geometric cluster visualization.
- **Phase 7: Hierarchical Clustering & Dendrogram:** Agglomerative clustering with Ward's linkage.
- **Phase 8: RFM (Recency, Frequency, Monetary) Modeling:** Quantile-based CRM segmentation.
- **Phase 9: Strategic Persona Development & Business Recommendations:** Actionable playbooks for each customer segment.


## 1. Environment Setup & Library Imports

In [ ]:
import os
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Metrics
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

# Set visual styles
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

print("✅ Analytics environment loaded successfully!")


## 2. Data Ingestion & Data Hygiene Audit

In [ ]:
# Ingestion from primary CSV (with SQLite fallback)
csv_path = "../data/customer_segmentation_data.csv" if os.path.exists("../data/customer_segmentation_data.csv") else "data/customer_segmentation_data.csv"
db_path = "../data/customer_segmentation.db" if os.path.exists("../data/customer_segmentation.db") else "data/customer_segmentation.db"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"📥 Loaded {len(df):,} records from CSV: {csv_path}")
elif os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM customers", conn)
    conn.close()
    print(f"📥 Loaded {len(df):,} records from SQLite Database: {db_path}")
else:
    raise FileNotFoundError("Customer dataset not found.")

# Display basic structural dimensions
print(f"Shape: {df.shape[0]} Rows, {df.shape[1]} Columns")
df.head(5)


In [ ]:
# Checking data types, nulls, and duplicate records
print("--- Data Information ---")
print(df.info())

print("\n--- Missing Values Audit ---")
print(df.isnull().sum())

print(f"\nDuplicate Customer IDs: {df.duplicated(subset=['Customer_ID']).sum()}")


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive numerical statistics
df.describe().T.round(2)


### 3.1 Demographic & Behavioral Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Age Distribution
sns.histplot(df["Age"], kde=True, ax=axes[0, 0], color="#2563EB", bins=20)
axes[0, 0].set_title("Customer Age Distribution")

# 2. Annual Income
sns.histplot(df["Annual_Income_k"], kde=True, ax=axes[0, 1], color="#16A34A", bins=20)
axes[0, 1].set_title("Annual Income Distribution ($k)")

# 3. Spending Score
sns.histplot(df["Spending_Score"], kde=True, ax=axes[0, 2], color="#9333EA", bins=20)
axes[0, 2].set_title("Spending Score Distribution (1-100)")

# 4. Purchase Frequency
sns.histplot(df["Purchase_Frequency"], kde=True, ax=axes[1, 0], color="#EA580C", bins=20)
axes[1, 0].set_title("Annual Purchase Frequency")

# 5. Recency Days
sns.histplot(df["Recency_Days"], kde=True, ax=axes[1, 1], color="#DC2626", bins=20)
axes[1, 1].set_title("Recency (Days Since Last Order)")

# 6. Annual Spend
sns.histplot(df["Annual_Spend"], kde=True, ax=axes[1, 2], color="#0D9488", bins=20)
axes[1, 2].set_title("Annual Total Spend ($)")

plt.tight_layout()
plt.show()


### 3.2 Categorical Channel & Preference Dynamics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Category Preferences
sns.countplot(data=df, y="Preferred_Category", ax=axes[0, 0], order=df["Preferred_Category"].value_counts().index, palette="Blues_r")
axes[0, 0].set_title("Customer Volume by Preferred Product Category")

# Shopping Channel
sns.countplot(data=df, x="Preferred_Channel", ax=axes[0, 1], palette="Greens_r")
axes[0, 1].set_title("Shopping Channel Preference")

# Payment Mode
sns.countplot(data=df, x="Payment_Method", ax=axes[1, 0], palette="Purples_r")
axes[1, 0].set_title("Preferred Payment Method")

# Discount Sensitivity
sns.countplot(data=df, x="Discount_Sensitivity", ax=axes[1, 1], palette="Oranges_r", order=["Low", "Medium", "High"])
axes[1, 1].set_title("Customer Discount Sensitivity")

plt.tight_layout()
plt.show()


### 3.3 Correlation Analysis

In [ ]:
num_cols = ["Age", "Annual_Income_k", "Spending_Score", "Annual_Spend", "Purchase_Frequency", "Recency_Days", "Average_Order_Value", "Customer_Lifetime_Value"]
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Correlation Matrix of Customer Metrics", fontsize=14, fontweight="bold")
plt.show()


## 4. Feature Selection & Standardization
For unsupervised clustering, distance-based algorithms such as K-Means and Agglomerative Hierarchical Clustering are sensitive to feature scales.
We standardize our core behavioral and demographic features using `StandardScaler` so each feature has $\mu = 0$ and $\sigma = 1$.


In [ ]:
clustering_features = ["Age", "Annual_Income_k", "Spending_Score", "Purchase_Frequency", "Recency_Days"]
X = df[clustering_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

print("Standardized Feature Matrix Shape:", X_scaled.shape)
X_scaled_df.head(3)


## 5. Optimal Cluster Evaluation: Elbow Method & Silhouette Analysis
To determine the optimal number of clusters ($k$), we evaluate two complementary statistical techniques:
1. **Elbow Method (Inertia / WCSS):** Plots the Within-Cluster Sum of Squares across $k \in [2, 10]$ to locate the point where the marginal reduction in inertia diminishes ("the elbow").
2. **Silhouette Coefficient Analysis:** Measures how similar an object is to its own cluster compared to neighboring clusters. Scores range from -1 (poor) to +1 (ideal).


In [ ]:
k_range = range(2, 11)
wcss = []
silhouette_scores = []
db_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    score = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(score)
    db_scores.append(davies_bouldin_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Curve
ax1.plot(k_range, wcss, marker='o', linewidth=2, color='#2563EB')
ax1.axvline(x=5, color='#DC2626', linestyle='--', label='Optimal k = 5')
ax1.set_title("Elbow Method (WCSS vs. Number of Clusters)", fontweight="bold")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Within-Cluster Sum of Squares (Inertia)")
ax1.legend()
ax1.grid(True)

# Silhouette Scores
ax2.plot(k_range, silhouette_scores, marker='s', linewidth=2, color='#16A34A')
ax2.axvline(x=5, color='#DC2626', linestyle='--', label='Optimal k = 5')
ax2.set_title("Silhouette Coefficient vs. Number of Clusters", fontweight="bold")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Average Silhouette Score")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Optimal k determined: 5 clusters (Peak Silhouette Score = {silhouette_scores[3]:.4f})")


## 6. Model Training: K-Means Clustering

In [ ]:
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=15)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Compute Silhouette score on final model
final_sil_score = silhouette_score(X_scaled, df["Cluster"])
final_db_score = davies_bouldin_score(X_scaled, df["Cluster"])
print(f"✅ K-Means Model Fitted Successfully!")
print(f"Silhouette Score: {final_sil_score:.4f}")
print(f"Davies-Bouldin Index: {final_db_score:.4f} (lower is better)")


### 6.1 Cluster Persona Mapping & Interpretation

In [ ]:
cluster_profile_raw = df.groupby("Cluster")[["Annual_Income_k", "Spending_Score", "Age", "Purchase_Frequency", "Recency_Days", "Annual_Spend"]].mean()

persona_names = {}
for c_idx, row in cluster_profile_raw.iterrows():
    inc = row["Annual_Income_k"]
    score = row["Spending_Score"]
    if inc > 75 and score > 65:
        persona_names[c_idx] = "Elite VIP Champions"
    elif inc > 75 and score <= 45:
        persona_names[c_idx] = "Affluent Savers"
    elif inc <= 55 and score > 65:
        persona_names[c_idx] = "Young Trendsetters"
    elif inc <= 50 and score <= 45:
        persona_names[c_idx] = "Bargain Seekers"
    else:
        persona_names[c_idx] = "Core Balanced Families"

df["Cluster_Persona"] = df["Cluster"].map(persona_names)

print("Cluster Sizes:")
print(df["Cluster_Persona"].value_counts())


### 6.2 Visualizing Clusters: Annual Income vs. Spending Score

In [ ]:
plt.figure(figsize=(12, 7))
palette = ["#2563EB", "#16A34A", "#9333EA", "#EA580C", "#0D9488"]

sns.scatterplot(
    data=df,
    x="Annual_Income_k",
    y="Spending_Score",
    hue="Cluster_Persona",
    palette=palette,
    s=75,
    alpha=0.85,
    edgecolor="none"
)

# Centroids projection in original feature space
centroids_scaled = kmeans.cluster_centers_
centroids_orig = scaler.inverse_transform(centroids_scaled)
inc_idx = clustering_features.index("Annual_Income_k")
score_idx = clustering_features.index("Spending_Score")

plt.scatter(
    centroids_orig[:, inc_idx],
    centroids_orig[:, score_idx],
    marker="X",
    s=250,
    color="black",
    edgecolor="yellow",
    linewidth=2,
    label="Cluster Centroids"
)

plt.title("Customer Segments: Annual Income ($k) vs. Spending Score (1-100)", fontsize=14, fontweight="bold")
plt.xlabel("Annual Income ($k)")
plt.ylabel("Spending Score (1-100)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Segments")
plt.tight_layout()
plt.show()


## 7. Dimensionality Reduction via Principal Component Analysis (PCA)
While the 2D plot of Income vs Spending Score is highly intuitive, customer behavior spans 5 standardized dimensions.
We perform **PCA** to capture the maximum multi-dimensional variance into 2 orthogonal components.


In [ ]:
pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(X_scaled)
df["PCA1"] = pca_transformed[:, 0]
df["PCA2"] = pca_transformed[:, 1]

var_ratio = pca.explained_variance_ratio_
print(f"Explained Variance Ratio: PC1 = {var_ratio[0]*100:.2f}%, PC2 = {var_ratio[1]*100:.2f}% (Total = {sum(var_ratio)*100:.2f}%)")

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="Cluster_Persona",
    palette=palette,
    s=80,
    alpha=0.85
)
plt.title(f"2D PCA Projection of Customer Clusters (Variance Explained: {sum(var_ratio)*100:.1f}%)", fontsize=14, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({var_ratio[0]*100:.1f}% var)")
plt.ylabel(f"Principal Component 2 ({var_ratio[1]*100:.1f}% var)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Segment")
plt.tight_layout()
plt.show()


## 8. Hierarchical / Agglomerative Clustering & Dendrogram
Hierarchical clustering builds a tree of clusters without requiring an a priori definition of $k$.
Using **Ward's minimum variance method**, we visualize the hierarchical merge distances via a **Dendrogram**.


In [ ]:
# Sample 150 points for an interpretable, clean dendrogram display
sample_indices = np.random.RandomState(42).choice(len(X_scaled), size=150, replace=False)
X_sample = X_scaled[sample_indices]

linkage_matrix = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=90.,
    leaf_font_size=10.,
    show_contracted=True
)
plt.title("Hierarchical Clustering Dendrogram (Ward Linkage)", fontsize=14, fontweight="bold")
plt.xlabel("Cluster Group / Sample Index")
plt.ylabel("Euclidean Distance Threshold")
plt.axhline(y=12.0, color='r', linestyle='--', label='Cut Threshold for 5 Clusters')
plt.legend()
plt.tight_layout()
plt.show()


## 9. RFM (Recency, Frequency, Monetary) CRM Segmentation
In addition to machine learning clustering, enterprise CRM systems utilize **RFM Segmentation**:
- **Recency (R):** How recently did the customer purchase? (Lower days = higher score)
- **Frequency (F):** How frequently do they purchase? (Higher orders = higher score)
- **Monetary (M):** How much do they spend annually? (Higher spend = higher score)


In [ ]:
# Assign quintile scores (1 to 5)
df["R_Score"] = pd.qcut(df["Recency_Days"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
df["F_Score"] = pd.qcut(df["Purchase_Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df["M_Score"] = pd.qcut(df["Annual_Spend"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df["RFM_Composite"] = df["R_Score"].astype(str) + df["F_Score"].astype(str) + df["M_Score"].astype(str)

def assign_rfm_label(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif f >= 4 and m >= 3:
        return "Loyal Customers"
    elif r >= 4 and f in [2, 3]:
        return "Potential Loyalists"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r in [2, 3] and f in [2, 3]:
        return "Need Attention"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r == 1 and f >= 4:
        return "Can't Lose Them"
    else:
        return "Hibernating / Churned"

df["RFM_Segment"] = df.apply(assign_rfm_label, axis=1)

# Summary of RFM Segments
rfm_summary = df.groupby("RFM_Segment").agg(
    Customer_Count=("Customer_ID", "count"),
    Avg_Recency_Days=("Recency_Days", "mean"),
    Avg_Frequency=("Purchase_Frequency", "mean"),
    Total_Revenue=("Annual_Spend", "sum"),
    Avg_Spend=("Annual_Spend", "mean")
).reset_index().sort_values(by="Total_Revenue", ascending=False)

rfm_summary.round(2)


### 9.1 RFM Segment Revenue & Customer Share

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Count
sns.barplot(data=rfm_summary, y="RFM_Segment", x="Customer_Count", palette="crest", ax=ax1)
ax1.set_title("Customer Count by RFM Segment", fontweight="bold")
ax1.set_xlabel("Number of Customers")
ax1.set_ylabel("RFM Segment")

# Revenue
sns.barplot(data=rfm_summary, y="RFM_Segment", x="Total_Revenue", palette="viridis", ax=ax2)
ax2.set_title("Total Revenue ($) by RFM Segment", fontweight="bold")
ax2.set_xlabel("Total Annual Spend ($)")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()


## 10. Customer Persona Profiling & Strategic Business Recommendations
Combining demographic variables, purchase patterns, and algorithmically derived clusters allows us to formulate targeted marketing strategies for each segment:


In [ ]:
cluster_stats = df.groupby("Cluster_Persona").agg(
    Count=("Customer_ID", "count"),
    Mean_Age=("Age", "mean"),
    Mean_Income_k=("Annual_Income_k", "mean"),
    Mean_Spend_Score=("Spending_Score", "mean"),
    Mean_Annual_Spend=("Annual_Spend", "mean"),
    Mean_Frequency=("Purchase_Frequency", "mean"),
    Mean_Recency=("Recency_Days", "mean"),
    Top_Category=("Preferred_Category", lambda x: x.mode()[0]),
    Top_Channel=("Preferred_Channel", lambda x: x.mode()[0]),
    Top_Payment=("Payment_Method", lambda x: x.mode()[0])
).reset_index()

cluster_stats["Customer_Share_%"] = (cluster_stats["Count"] / len(df)) * 100
cluster_stats["Revenue_Share_%"] = (df.groupby("Cluster_Persona")["Annual_Spend"].sum().values / df["Annual_Spend"].sum()) * 100

cluster_stats.round(2)


### 10.1 Strategic Marketing Playbook Matrix

| Cluster Segment | Key Characteristics | Preferred Categories & Channels | Strategic Objective | Actionable Marketing Campaign |
| :--- | :--- | :--- | :--- | :--- |
| **Elite VIP Champions** | High Income (\$108k), High Spend (\$15k+), Frequent (36 orders/yr), Recent (< 20 days) | Electronics & Apparel, Mobile App & Web | **Retention & Maximizing Lifetime Value** | Dedicated VIP concierge, invite-only luxury product previews, premium loyalty rewards, zero-fee expedited shipping. |
| **Affluent Savers** | High Income (\$102k), Low Spend (\$2.5k), Infrequent (8 orders/yr), High Savings | Home & Living, Electronics, Desktop Web | **Activation & Upselling** | Premium quality assurance marketing, exclusive high-ticket bundle offers, financial/warranty perks, personalized product curation. |
| **Young Trendsetters** | Younger (Age ~26), Moderate Income (\$46k), High Spend (\$6.5k), Very High Mobile App usage | Fashion & Beauty, Mobile App, Digital Wallets | **Engagement & Social Amplification** | Influencer partnerships, TikTok/Instagram push notifications, limited-edition flash drops, Buy Now Pay Later (BNPL) checkout. |
| **Bargain Seekers** | Lower-to-Mid Income (\$32k), Low Spend (\$1.2k), High Discount Sensitivity | Groceries & Basics, In-Store & Web | **Basket Building & Frequency** | Volume clearance discounts, digital coupon circulars, combo saver packs, price-match guarantees, free shipping thresholds. |
| **Core Balanced Families** | Middle Age (42), Middle Income (\$62k), Moderate Spend (\$4.8k), Steady monthly shopper | Groceries, Home & Living, Multi-channel | **Cross-Selling & Habit Formation** | Subscribe-and-Save replenishment discounts, family bundle kits, seasonal back-to-school / holiday loyalty promotions. |


## 11. Conclusion & Key Takeaways
1. **Mathematical Validation:** Both the Elbow curve and Silhouette analysis validated **$k = 5$** as the optimal cluster configuration.
2. **Revenue Concentration:** The **Elite VIP Champions** account for over **40% of total revenue** while representing only ~22% of the customer base. Protecting and nurturing this cohort is paramount to business resilience.
3. **Growth Opportunity in Affluent Savers:** High-income cautious savers represent an untapped high-margin opportunity if incentivized with premium quality positioning rather than generic discounting.
4. **Channel Alignment:** Young Trendsetters are heavily digital-first (Mobile App & UPI payments), whereas Core Families utilize balanced omnichannel touchpoints.
